In [4]:
using Distributions
using MCMCChains
using MLDataUtils: rescale!, shuffleobs, stratifiedobs
using Plots
using Random
using RDatasets
using StatsFuns: logistic
using StatsPlots
using Turing

Random.seed!(414)
Turing.turnprogress(false)

┌ Info: [Turing]: global PROGRESS is set as false
└ @ Turing /Users/dsp/.julia/packages/Turing/LONxt/src/Turing.jl:25


false

In [5]:
data = RDatasets.dataset("ISLR", "Default")
first(data, 6)

,Default,Student,Balance,Income
,Categorical…,Categorical…,Float64,Float64
1,No,No,729.526,44361.6
2,No,Yes,817.18,12106.1
3,No,No,1073.55,31767.1
4,No,No,529.251,35704.5
5,No,No,785.656,38463.5
6,No,Yes,919.589,7491.56


In [12]:
# Y/N = 1/0
data[!, :Default] = 1. .* (data[!, :Default] .== "Yes")
data[!, :Student] = 1. .* (data[!, :Student] .== "Yes")
first(data, 6)

,Default,Student,Balance,Income
,Float64,Float64,Float64,Float64
1,0.0,0.0,729.526,44361.6
2,0.0,0.0,817.18,12106.1
3,0.0,0.0,1073.55,31767.1
4,0.0,0.0,529.251,35704.5
5,0.0,0.0,785.656,38463.5
6,0.0,0.0,919.589,7491.56


In [13]:
function splitdata(df, target; at=0.7)
    shuffled = shuffleobs(df)
    train, test = stratifiedobs(row -> row[target], shuffled, p=at)
end

splitdata (generic function with 1 method)

In [15]:
features = [:Student, :Balance, :Income]
numerics = [:Balance, :Income]
target = :Default
train, test = splitdata(data, target, at=0.05);

In [16]:
for feature in numerics
    mu, sig = rescale!(train[!, feature], obsdim=1)
    rescale!(test[!, feature], mu, sig, obsdim=1)
end

In [18]:
Xtrain = Matrix(train[:, features])
Xtest = Matrix(test[:, features])
ytrain = train[:, target]
ytest = test[:, target];